# Populate Tier-1 feature cache (ESM-2, AlphaFold-DB, MACE-OFF)

**Run on Colab** with a GPU (A100 / L4 / T4 / Blackwell Pro all work).
Populates the three Tier-1 parquet files under
`cell_sim/features/cache/`:

| Extractor        | Status on a JCVI-Syn3A run                          |
|------------------|-----------------------------------------------------|
| ESM-2 (650M)     | full per-protein 1280-dim embeddings, ~18 s on A100 |
| AlphaFold-DB     | empty — JCVI-Syn3A (taxid 2144189) is a synthetic organism and UniProt does not index its proteome, so AFDB has nothing to fetch. Session 15 will wire in a curated mapping from the Luthey-Schulten supplement. |
| MACE-OFF (small) | empty — the Syn3A SBML encodes metabolites as species IDs (`M_atp_c`), not SMILES. Session 15 will wire in a curated SMILES map (KEGG / ChEBI) before invoking the backend. |

So: **ESM-2 is the real deliverable this session.** The other two
parquets are registered + tracked by the manifest but carry no
information content yet. That keeps the three-source manifest
consistent for Session 15 without manufacturing fake data.

Non-goals (deliberately): no detector changes, no MCC measurement,
no model weights committed to the repo.


In [ ]:
# Cell 2 — install the GPU-side deps. These are intentionally NOT in
# cell_sim/requirements.txt — see the comment there.
#
# The install is split into two groups:
#   1. Core stack (ESM-2 + AlphaFold + framework). Required.
#   2. MACE-OFF stack. Optional — mace-torch pins an older e3nn, so
#      we let pip resolve mace's own compatible version and tolerate
#      a failure (ESM-2 + AlphaFold still populate).

# --- Group 1: core stack (ESM-2, AlphaFold, supporting libs) ---
!pip install -q \
    "torch>=2.2" \
    "transformers>=4.40" \
    "biopython>=1.83" \
    "rdkit>=2023.9" \
    "ase>=3.22" \
    "pyarrow>=14" \
    "pandas>=2" \
    "numpy>=1.24"

# --- Group 2: MACE-OFF (optional, may fail; ESM-2 / AlphaFold survive) ---
# NOTE: do NOT pin e3nn here — mace-torch's own metadata selects a
# compatible version. Pinning e3nn>=0.5.1 breaks resolution on
# every mace-torch 0.3.x release (they pin e3nn<0.5).
import subprocess, sys
_mace_status = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "mace-torch"],
    capture_output=True, text=True,
)
MACE_AVAILABLE = (_mace_status.returncode == 0)
if not MACE_AVAILABLE:
    print("[warn] mace-torch install failed; MACE-OFF cell will be skipped.")
    print("       last pip stderr line:")
    for line in _mace_status.stderr.strip().splitlines()[-3:]:
        print(f"         {line}")
else:
    print("mace-torch install OK")

import torch
print(f"\ntorch {torch.__version__}  cuda={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"  VRAM: {props.total_memory / 1024**3:.1f} GiB")
else:
    print("  CPU only — ESM-2 will run but will take much longer.")
print(f"MACE_AVAILABLE = {MACE_AVAILABLE}")


In [ ]:
# Cell 3 — clone the project repo at the current Session-13 HEAD.
# If you've already cloned into /content/cell, the second command
# just switches into it without re-cloning.
import os
if not os.path.isdir("/content/cell"):
    !git clone --branch claude/syn3a-whole-cell-simulator-REjHC \
        https://github.com/Nikku03/cell.git /content/cell
%cd /content/cell

import sys
sys.path.insert(0, "/content/cell")
print("CWD:", os.getcwd())
!git log --oneline -1


In [ ]:
# Cell 4 — stage the Luthey-Schulten input data, then load Syn3A CDS
# sequences from the staged GenBank file.
#
# The 5 upstream files (syn3A.gb, kinetic_params.xlsx,
# initial_concentrations.xlsx, complex_formation.xlsx, Syn3A_updated.xml)
# are NOT committed to Nikku03/cell — they live under
# cell_sim/data/Minimal_Cell_ComplexFormation/input_data/ which is
# gitignored (see memory_bank/data/STAGING.md). We fetch them from
# the upstream repo here, idempotent (skipped if already present).
import urllib.request
from pathlib import Path

STAGED_DIR = Path(
    "cell_sim/data/Minimal_Cell_ComplexFormation/input_data"
)
STAGED_DIR.mkdir(parents=True, exist_ok=True)
UPSTREAM = (
    "https://raw.githubusercontent.com/Luthey-Schulten-Lab/"
    "Minimal_Cell_ComplexFormation/master/input_data"
)
STAGED_FILES = [
    "syn3A.gb",
    "kinetic_params.xlsx",
    "initial_concentrations.xlsx",
    "complex_formation.xlsx",
    "Syn3A_updated.xml",
]
for fname in STAGED_FILES:
    dest = STAGED_DIR / fname
    if dest.exists() and dest.stat().st_size > 0:
        continue
    url = f"{UPSTREAM}/{fname}"
    print(f"staging {fname} from upstream...")
    urllib.request.urlretrieve(url, dest)
print(f"staged {len(STAGED_FILES)} files under {STAGED_DIR}")

# --- load Syn3A CDS translations ---
from Bio import SeqIO
import pandas as pd

GB_PATH = STAGED_DIR / "syn3A.gb"
records = list(SeqIO.parse(GB_PATH, "genbank"))
assert len(records) == 1, f"expected 1 record, got {len(records)}"
rec = records[0]
print(f"accession={rec.id}  length={len(rec.seq):,} bp")

cds_rows = []
for feat in rec.features:
    if feat.type != "CDS":
        continue
    locus = (feat.qualifiers.get("locus_tag") or [""])[0]
    gene_name = (feat.qualifiers.get("gene") or [""])[0]
    translation = (feat.qualifiers.get("translation") or [""])[0]
    protein_id = (feat.qualifiers.get("protein_id") or [""])[0]
    product = (feat.qualifiers.get("product") or [""])[0]
    if not locus or not translation:
        continue
    cds_rows.append({
        "locus_tag": locus,
        "gene_name": gene_name,
        "protein_id": protein_id,
        "product": product,
        "sequence": translation.upper(),
    })
cds_df = pd.DataFrame(cds_rows)
print(f"\n{len(cds_df)} CDS with translations")
print(cds_df.head(3))
dnaA_match = cds_df.loc[cds_df["gene_name"] == "dnaA"]
if len(dnaA_match):
    dnaA_seq = dnaA_match["sequence"].iloc[0]
    print(f"\ndnaA length: {len(dnaA_seq)} aa; starts: {dnaA_seq[:10]}")


In [ ]:
# Cell 5 — ESM-2 650M embeddings for every CDS with a translation.
import hashlib, time
from pathlib import Path
from cell_sim.features.extractors import ESM2Extractor
from cell_sim.features.batched_inference import (
    BatchedInferenceConfig,
)

CACHE_DIR = Path("cell_sim/features/cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

esm = ESM2Extractor()
cfg = BatchedInferenceConfig(
    batch_size=16,
    device="auto",
    dtype="float16" if torch.cuda.is_available() else "float32",
    max_seq_length=1022,
    progress=True,
)

t0 = time.perf_counter()
esm_feats = esm.extract(cds_df[["locus_tag", "sequence"]], cfg)
esm_wall = time.perf_counter() - t0
esm_path, esm_sha = esm.write_cache(esm_feats, cache_dir=CACHE_DIR)
print(f"ESM-2 done in {esm_wall:.1f}s  parquet={esm_path}")
print(f"  rows={len(esm_feats)}  dims={len(esm.feature_cols)}")
print(f"  sha256={esm_sha}")
print("\nfirst 5 locus tags' embedding L2 norms:")
for locus in esm_feats.index[:5]:
    vec = esm_feats.loc[locus]
    print(f"  {locus}  ||x||={(vec ** 2).sum() ** 0.5:.3f}")


In [ ]:
# Cell 6 — AlphaFold-DB per-protein structural descriptors.
#
# JCVI-Syn3A (taxid 2144189, "synthetic bacterium JCVI-Syn3A") is not
# indexed in UniProt as of this writing. The AlphaFold-DB URL scheme
# is AF-<UniProt>-F1-model_v4.pdb, so without a UniProt mapping we
# have nothing to fetch. We STILL write a valid parquet with one
# NaN row per CDS + af_has_structure = 0.0 so the manifest has an
# entry for this source and Session 15 can swap in real values
# without touching the registry contract.
#
# If a future run lands a UniProt mapping (e.g. via Luthey-Schulten's
# supplement), the Route-D' stream below will populate af_input
# and the real AlphaFoldExtractor.extract() path runs automatically.
import json as _json, time, urllib.request
import numpy as np
from pathlib import Path
from cell_sim.features.extractors import AlphaFoldExtractor

UNIPROT_API = "https://rest.uniprot.org"
CACHE_DIR = Path("cell_sim/features/cache")

# Derive Syn3A taxid from the GenBank source feature.
src_feat = [f for f in rec.features if f.type == "source"][0]
taxid_tokens = [x.split(":", 1)[1]
                for x in src_feat.qualifiers.get("db_xref", [])
                if x.startswith("taxon:")]
syn3a_taxid = taxid_tokens[0] if taxid_tokens else None
print(f"Syn3A taxid from GenBank source feature: {syn3a_taxid}")

ncbi_to_uniprot: dict[str, str] = {}
if syn3a_taxid:
    url = (f"{UNIPROT_API}/uniprotkb/stream?"
           f"query=organism_id:{syn3a_taxid}"
           "&format=tsv&fields=accession,xref_embl")
    try:
        req = urllib.request.Request(
            url,
            headers={"User-Agent": "cell_sim-populate_tier1_cache/0.1"},
        )
        with urllib.request.urlopen(req, timeout=60) as resp:
            txt = resp.read().decode("utf-8")
        for line in txt.splitlines()[1:]:
            fields = line.split("\t")
            if len(fields) < 2:
                continue
            acc = fields[0]
            for tok in fields[1].replace(";", " ").split():
                tok = tok.strip().rstrip(";")
                if tok.startswith("AVX") and tok[3:].split(".")[0].isdigit():
                    ncbi_to_uniprot[tok.split(".")[0]] = acc
    except Exception as exc:  # noqa: BLE001 — network failures aren't fatal
        print(f"  UniProt stream query failed: {type(exc).__name__}: {exc}")

print(f"UniProt accessions found for taxid {syn3a_taxid}: "
      f"{len(ncbi_to_uniprot)}")

af = AlphaFoldExtractor()
af_input = cds_df[["locus_tag", "protein_id"]].copy()
af_input["uniprot_id"] = (
    af_input["protein_id"].str.split(".").str[0]
    .map(ncbi_to_uniprot).fillna("")
)
n_mapped = int((af_input["uniprot_id"] != "").sum())
print(f"UniProt IDs mapped: {n_mapped}/{len(cds_df)} CDS")

t0 = time.perf_counter()
if n_mapped == 0:
    # Build an empty-but-valid parquet so the manifest has a slot.
    af_feats = pd.DataFrame(
        {c: [np.nan] * len(cds_df) for c in af.feature_cols},
    )
    af_feats["af_has_structure"] = 0.0
    af_feats.index = pd.Index(cds_df["locus_tag"].tolist(), name="locus_tag")
    print("  no UniProt mapping available for JCVI-Syn3A — writing an")
    print("  empty parquet (all NaN, af_has_structure=0). Session 15 TODO.")
else:
    af_feats = af.extract(af_input[["locus_tag", "uniprot_id"]])
af_wall = time.perf_counter() - t0
af_path, af_sha = af.write_cache(af_feats, cache_dir=CACHE_DIR)
n_struct = int(af_feats["af_has_structure"].sum())
print(f"AlphaFold done in {af_wall:.1f}s  parquet={af_path}")
print(f"  rows={len(af_feats)}  structures found={n_struct}")
print(f"  sha256={af_sha}")


In [ ]:
# Cell 7 — MACE-OFF BDE-derived k_cat refinement.
#
# The Syn3A SBML encodes metabolites as BiGG-style species IDs
# (M_atp_c, M_h2o_c, ...), NOT SMILES. MACE-OFF needs real SMILES
# to compute bond-dissociation energies. Until we wire in a curated
# species-ID -> SMILES map (KEGG / ChEBI / ChemAxon — Session 15
# TODO), passing the species IDs to the backend produces ~thousands
# of RDKit parse errors and a parquet of mostly-NaN rows. Cleaner
# to write an empty parquet that the manifest still tracks.
#
# When a real SMILES map lands, this cell's else-branch runs the
# full per-reaction bootstrap via MaceOffExtractor.
import time
import numpy as np
from cell_sim.features.extractors import MaceOffExtractor

mace = MaceOffExtractor(model="small", device="auto")

# Always writes an empty parquet in this session — flip to True once
# a curated SMILES map is available and SUBSTRATE_SMILES_MAP is
# populated.
HAVE_CURATED_SMILES_MAP = False

t0 = time.perf_counter()
if not HAVE_CURATED_SMILES_MAP or not MACE_AVAILABLE:
    if not MACE_AVAILABLE:
        reason = "mace-torch install failed (see cell 2 pip output)"
    else:
        reason = "no curated SBML-species -> SMILES map yet (Session 15 TODO)"
    print(f"skipping MACE-OFF bootstrap: {reason}")
    mace_feats = pd.DataFrame(
        {c: [np.nan] * len(cds_df) for c in mace.feature_cols},
    )
    mace_feats["mace_has_estimate"] = 0.0
    mace_feats.index = pd.Index(cds_df["locus_tag"].tolist(), name="locus_tag")
else:
    from cell_sim.layer3_reactions.sbml_parser import parse_sbml
    SBML_PATH = Path(
        "cell_sim/data/Minimal_Cell_ComplexFormation/input_data/Syn3A_updated.xml"
    )
    sbml = parse_sbml(SBML_PATH)
    mace_rows = []
    for rxn in sbml.reactions.values():
        if not rxn.gene_associations:
            continue
        for gene_id in rxn.gene_associations:
            locus = gene_id.replace("G_MMSYN1_", "JCVISYN3A_")
            for sp_id, _ in rxn.reactants.items():
                # SUBSTRATE_SMILES_MAP[sp_id] would be the real SMILES;
                # until a curated map lands this branch is unreachable.
                smi = SUBSTRATE_SMILES_MAP.get(sp_id)  # type: ignore[name-defined]
                if not smi:
                    continue
                mace_rows.append({
                    "locus_tag": locus,
                    "enzyme_name": rxn.short_name,
                    "reaction_class": "metabolic",
                    "substrate_smiles": smi,
                })
    mace_input = pd.DataFrame(mace_rows)
    print(f"MACE inputs: {len(mace_input)} (enzyme, substrate) pairs")
    mace_feats = mace.extract(mace_input)

mace_wall = time.perf_counter() - t0
mace_path, mace_sha = mace.write_cache(mace_feats, cache_dir=CACHE_DIR)
n_est = int(mace_feats["mace_has_estimate"].sum())
print(f"MACE-OFF done in {mace_wall:.1f}s  parquet={mace_path}")
print(f"  rows={len(mace_feats)}  estimates={n_est}")
print(f"  sha256={mace_sha}")


In [ ]:
# Cell 8 — refresh manifest.json. write_cache() already added
# each parquet, but we re-load and re-verify here so the user
# sees the full manifest state in one place.
from cell_sim.features.cache_manifest import CachedFeatureManifest

MANIFEST_PATH = CACHE_DIR / "manifest.json"
manifest = CachedFeatureManifest.load(MANIFEST_PATH)
print(f"manifest at {MANIFEST_PATH}")
print(f"  tracked sources: {sorted(manifest.sources)}")
for name, entry in sorted(manifest.sources.items()):
    ok = manifest.verify(name, CACHE_DIR / f"{name}.parquet")
    print(f"  {name:20s}  v{entry['version']}  rows={entry['rows']:<6}  "
          f"sha256={entry['sha256'][:12]}...  verified={ok}")


In [ ]:
# Cell 9 — choose how to ship the populated cache back to your repo
# or machine. Three modes supported:
#
#   "download"    — four browser-download prompts (default; safest)
#   "drive"       — copies to /content/drive/MyDrive/cell_sim_cache/
#   "github_pat"  — fully automated git add -f + commit + push
#
# None of the modes raise on misconfiguration; a missing PAT or Drive
# denial prints clear instructions and leaves the cache intact so
# you can flip modes and re-run without recomputing anything.

OUTPUT_MODE = "download"   # options: "download" | "drive" | "github_pat"

PARQUET_NAMES = ("esm2_650M", "alphafold_db", "mace_off_kcat")
PARQUET_PATHS = [CACHE_DIR / f"{name}.parquet" for name in PARQUET_NAMES]


def _do_download() -> None:
    from google.colab import files  # type: ignore
    for p in PARQUET_PATHS:
        files.download(str(p))
    files.download(str(MANIFEST_PATH))
    print("Four downloads triggered (3 parquets + manifest.json).")


def _do_drive() -> None:
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
    except Exception as exc:  # noqa: BLE001
        print(f"Drive mount failed: {type(exc).__name__}: {exc}")
        print("Flip OUTPUT_MODE to 'download' or 'github_pat' and rerun.")
        return
    import shutil
    dest = Path("/content/drive/MyDrive/cell_sim_cache")
    dest.mkdir(parents=True, exist_ok=True)
    for p in PARQUET_PATHS:
        shutil.copy(p, dest / p.name)
    shutil.copy(MANIFEST_PATH, dest / "manifest.json")
    print(f"Copied 3 parquets + manifest.json to {dest}")


def _do_github_pat() -> None:
    import os, subprocess
    pat = os.environ.get("GITHUB_PAT", "").strip()
    if not pat:
        print("GITHUB_PAT is not set in this kernel. Either:")
        print("  1. Colab left sidebar -> Secrets -> add 'GITHUB_PAT',")
        print("     then run:")
        print("       from google.colab import userdata")
        print("       import os")
        print("       os.environ['GITHUB_PAT'] = userdata.get('GITHUB_PAT')")
        print("  2. Or inline: %env GITHUB_PAT=ghp_xxx")
        print("Then re-run this cell.")
        return

    def run(cmd: list[str], check: bool = True):
        res = subprocess.run(cmd, capture_output=True, text=True)
        if res.stdout.strip():
            print(res.stdout.rstrip())
        if res.stderr.strip():
            print(res.stderr.rstrip())
        if check and res.returncode != 0:
            raise RuntimeError(f"{cmd!r} exit {res.returncode}")
        return res

    run(["git", "config", "user.email", "cell-sim-bot@noreply.local"])
    run(["git", "config", "user.name", "cell-sim-bot"])
    # -f because the parquets are gitignored by default.
    paths = [str(p) for p in PARQUET_PATHS] + [str(MANIFEST_PATH)]
    run(["git", "add", "-f", *paths])
    status = run(["git", "status", "--porcelain"], check=False)
    if not status.stdout.strip():
        print("No changes to commit (parquets already at origin).")
        return
    run(["git", "commit", "-m",
         "Session 14: populate Tier 1 cache (ESM-2, AlphaFold-DB, MACE-OFF)"])
    remote = f"https://{pat}@github.com/Nikku03/cell.git"
    run(["git", "push", remote, "claude/syn3a-whole-cell-simulator-REjHC"])
    print("Push complete.")


if OUTPUT_MODE == "download":
    _do_download()
elif OUTPUT_MODE == "drive":
    _do_drive()
elif OUTPUT_MODE == "github_pat":
    _do_github_pat()
else:
    print(f"unknown OUTPUT_MODE={OUTPUT_MODE!r}; "
          "choose one of download | drive | github_pat")


In [ ]:
# Cell 10 — final summary. Paste the output back to the review
# thread so Session 15 can verify the SHAs before running the
# downstream detector stack.
from textwrap import dedent
manifest = CachedFeatureManifest.load(MANIFEST_PATH)
lines = ["## Tier-1 cache population summary", ""]
for name, entry in sorted(manifest.sources.items()):
    parquet = CACHE_DIR / f"{name}.parquet"
    df = pd.read_parquet(parquet)
    lines.append(f"### {name}  (v{entry['version']})")
    lines.append(f"- parquet: `{parquet}`")
    lines.append(f"- rows: {entry['rows']}")
    lines.append(f"- sha256: `{entry['sha256']}`")
    lines.append(f"- created_at: {entry['created_at']}")
    lines.append(
        f"- first 3 locus_tags: "
        f"{list(df.index[:3])}"
    )
    sample_cols = list(df.columns[:5])
    lines.append(f"- first 5 columns: {sample_cols}")
    if sample_cols:
        first_values = df.iloc[0][sample_cols].to_dict()
        lines.append(f"- first-row values: {first_values}")
    lines.append("")
wall_lines = [
    f"- ESM-2 wall:      {esm_wall:.1f}s",
    f"- AlphaFold wall:  {af_wall:.1f}s",
    f"- MACE-OFF wall:   {mace_wall:.1f}s",
]
lines.append("### Wall times")
lines.extend(wall_lines)
print("\n".join(lines))
